In [1]:
# import cv2
# import os

# def extract_frames_from_videos(video_folder, output_folder, frame_rate=10):
#     """
#     Extract frames from all videos in a folder.
#     """
#     # Get list of all .avi files in the folder
#     video_files = [f for f in os.listdir(video_folder) if f.endswith(".avi")]

#     for video_file in video_files:
#         video_path = os.path.join(video_folder, video_file)
#         # Create a folder for this video's frames
#         video_name = os.path.splitext(video_file)[0]
#         video_output_folder = os.path.join(output_folder, f"{video_name}_frames")
#         os.makedirs(video_output_folder, exist_ok=True)

#         # Open the video
#         cap = cv2.VideoCapture(video_path)
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         frame_interval = int(fps // frame_rate)  # Calculate frame interval
#         count = 0

#         while cap.isOpened():
#             ret, frame = cap.read()
#             if not ret:
#                 break
#             if count % frame_interval == 0:
#                 frame_name = f"{video_output_folder}/frame_{count:04d}.jpg"
#                 cv2.imwrite(frame_name, frame)  # Save frame
#             count += 1

#         cap.release()
#         print(f"Frames extracted for {video_file} in {video_output_folder}")


In [2]:
# import os

# # Define input directories
# input_dirs = {
#     "Normal state": "/kaggle/input/vibration-analysis/Normal state-20250307T135903Z-001/Normal state",
#     "Bearing fault": "/kaggle/input/vibration-analysis/Bearing Fault-20250307T135906Z-001/Bearing Fault",
#     "Unbalance weight": "/kaggle/input/vibration-analysis/Unbalance weight-20250307T135900Z-001/Unbalance weight",
# }

# # Define base output directory
# output_base_dir = "Dataset"

# # Ensure output directories exist and extract frames
# for state, input_path in input_dirs.items():
#     output_path = os.path.join(output_base_dir, state, "frames")
#     os.makedirs(output_path, exist_ok=True)  # Create directory if not exists
#     extract_frames_from_videos(input_path, output_path)

# print("Frame extraction completed successfully!")


Feature Extraction For Each Frame

In [3]:
# import cv2
# import os
# import numpy as np
# import pandas as pd

# base_path = '/kaggle/input/extracted-frames/Dataset'

# classes = {
#     'normal': [
#         'Normal state/frames/250 rpm front and distance 40 cm_frames',
#         'Normal state/frames/250 rpm with small angle_frames'
#     ],
#     'bearing_fault': [
#         'Bearing fault/frames/250 rpm front and distance 40 cm_frames',
#         'Bearing fault/frames/250 rpm with small angle_frames'
#     ],
#     'unbalanced_weight': [
#         'Unbalance weight/frames/250 rpm and front_frames',
#         'Unbalance weight/frames/250 rpm front and distance 40cm_frames'
#     ]
# }

# data = []

# img_size = (64, 64)

# # Function to calculate features for each frame
# def calculate_mean_intensity(frame_path):
#     frame = cv2.imread(frame_path)
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     return np.mean(gray)

# def calculate_edge_sharpness(frame_path):
#     frame = cv2.imread(frame_path)
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
#     sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
#     edge_sharpness = np.sqrt(sobel_x**2 + sobel_y**2)
#     return np.mean(edge_sharpness)

# def calculate_motion_between_frames(prev_frame_path, next_frame_path):
#     prev_frame = cv2.imread(prev_frame_path)
#     next_frame = cv2.imread(next_frame_path)
#     prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
#     next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)
#     flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
#     magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
#     return np.mean(magnitude)

# # Loop over classes and extract features for each frame
# for label, (class_name, folders) in enumerate(classes.items()):
#     for folder in folders:
#         folder_path = os.path.join(base_path, folder)
#         frame_files = sorted(os.listdir(folder_path))  # Ensure frames are sorted by name
        
#         # Iterate over frames
#         for i in range(len(frame_files) - 1):  # We need consecutive frames for motion
#             frame_path = os.path.join(folder_path, frame_files[i])
#             next_frame_path = os.path.join(folder_path, frame_files[i + 1])
            
#             # Extract features
#             mean_intensity = calculate_mean_intensity(frame_path)
#             edge_sharpness = calculate_edge_sharpness(frame_path)
#             motion = calculate_motion_between_frames(frame_path, next_frame_path)
            
#             # Store the extracted features and the corresponding label
#             feature_vector = [mean_intensity, edge_sharpness, motion]
#             data.append((feature_vector, label))  # Label is the index from the classes dictionary

# # Convert the data into a pandas DataFrame
# features = [item[0] for item in data]
# labels = [item[1] for item in data]
# df = pd.DataFrame(features, columns=["Mean Intensity", "Edge Sharpness", "Motion"])
# df["Label"] = labels

# # Display the DataFrame
# print(df.head())


In [4]:
# df.to_csv('extracted_features.csv', index=False)

In [5]:
import pandas as pd

df = pd.read_csv('/kaggle/input/extracted-features/extracted_features.csv')
print(df.head())


   Mean Intensity  Edge Sharpness    Motion  Label
0       90.884095       41.362134  0.763283      0
1       91.078851       38.748409  0.389510      0
2       96.036536       39.140079  0.287229      0
3      100.864997       39.916083  0.385493      0
4      106.160143       40.884420  0.356925      0


Normalise

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[["Mean Intensity", "Edge Sharpness", "Motion"]] = scaler.fit_transform(df[["Mean Intensity", "Edge Sharpness", "Motion"]])
print(df.head())

   Mean Intensity  Edge Sharpness    Motion  Label
0      -10.544595       -0.461284  3.941906      0
1      -10.463276       -1.320251  1.530277      0
2       -8.393223       -1.191534  0.870349      0
3       -6.377127       -0.936510  1.504360      0
4       -4.166169       -0.618278  1.320034      0


Convert to time series (10 frames per sequence)

In [7]:
import numpy as np

sequence_length = 10 
X = []
y = []

for label in df['Label'].unique():
    class_df = df[df['Label'] == label].reset_index(drop=True)

    for i in range(len(class_df) - sequence_length + 1):
        window = class_df.iloc[i:i+sequence_length][["Mean Intensity", "Edge Sharpness", "Motion"]].values
        X.append(window)
        y.append(label)

X = np.array(X)  
y = np.array(y) 

print("X shape:", X.shape)
print("y shape:", y.shape)

print("First sequence (X[0]):")
print(X[0])

print("\nCorresponding label (y[0]):", y[0])

X shape: (3980, 10, 3)
y shape: (3980,)
First sequence (X[0]):
[[-10.54459526  -0.4612838    3.94190605]
 [-10.46327607  -1.32025114   1.53027688]
 [ -8.39322279  -1.19153369   0.87034915]
 [ -6.37712679  -0.9365098    1.5043597 ]
 [ -4.16616863  -0.61827824   1.32003354]
 [ -3.19563097  -0.50278935   1.09365255]
 [ -2.56387369  -0.3798315    0.79654192]
 [ -2.01015859  -0.31690358   1.2379425 ]
 [ -1.45588894  -0.15400127   0.9434682 ]
 [ -1.22141716  -0.11255785   0.59719181]]

Corresponding label (y[0]): 0


Extract features using LSTM

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

feature_extractor = models.Sequential([
    layers.Input(shape=(10, 3)),  
    layers.LSTM(64, return_sequences=False)
])

X_features = feature_extractor.predict(X)  
print("X_features shape:", X_features.shape)

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
X_features shape: (3980, 64)


In [9]:
print("\nFirst 5 feature vectors:")
print(X_features[:5])


First 5 feature vectors:
[[-0.12281509 -0.35626093 -0.178653    0.05475847  0.22093509  0.17760566
   0.33109757  0.11229329 -0.01487989  0.14390148  0.0867357   0.3979009
  -0.00153163  0.05737641  0.17874442 -0.10264788  0.00452718  0.40334612
  -0.23900716 -0.04728211  0.02865895  0.13321412 -0.10359584  0.25045675
  -0.35027677  0.34789413  0.05382116 -0.31188196 -0.11055479  0.08555689
   0.17540947  0.0268473   0.19152716  0.10515112  0.20343167  0.17460743
  -0.14288889 -0.20988885  0.16236304  0.11594769  0.13837492 -0.31558847
  -0.04999922 -0.27363053  0.1502252  -0.02191654 -0.34782943  0.275941
   0.32520407 -0.06586003 -0.1513818  -0.13743803  0.22756943 -0.3581947
   0.279513   -0.19787662  0.18368016  0.21747895  0.14628653  0.10900473
   0.22754651  0.18561722  0.25247258 -0.19382381]
 [-0.12169363 -0.32694632 -0.175948    0.01422295  0.1796898   0.1891404
   0.24686727  0.12173641 -0.00622705  0.13361207  0.05704243  0.35264534
  -0.02028473  0.06027079  0.15720695 -0

Train SVM on LSTM Output

In [10]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42)

svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[270   1   0]
 [ 20 205   0]
 [  0   0 300]]
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       271
           1       1.00      0.91      0.95       225
           2       1.00      1.00      1.00       300

    accuracy                           0.97       796
   macro avg       0.98      0.97      0.97       796
weighted avg       0.98      0.97      0.97       796

